<a href="https://colab.research.google.com/github/sreent/data-management-intro/blob/main/Enforcement/05-lab-enforcement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Topic 5 Lab — Enforcement: Constraints, Access Control, Transactions, and Indexes

This lab builds a production-ready enforcement stack on the **Orders & Payments** schema from Chapters 2–4 (including the Office/Employee sales organisation extension from Chapter 4). The notebook is cumulative: each part depends on the previous, and you end with a hardened database. The focus is on enforcement — you do not redesign the schema; you harden it.

Use the formal vocabulary (constraint, CHECK, UNIQUE, NOT NULL, CASCADE, RESTRICT, view, GRANT, REVOKE, transaction, BEGIN, COMMIT, ROLLBACK, index, selectivity, EXPLAIN) throughout your work.

**Time budget:** Part 1 ~25 min | Part 2 ~20 min | Part 3 ~20 min | Part 4 ~20 min | Part 5 ~25 min | **Total ~110 min**

**Tools required:** Google Colab with MySQL (`%%sql` magic)

---

## Setup

Run the following cells in a new Google Colab notebook. The Orders & Payments schema is pre-loaded with primary keys and foreign keys — but deliberately without CHECK, extra UNIQUE, CASCADE, views, or indexes. This gives you a clean enforcement surface.

**Cell 1 — Install and connect:**

In [ ]:
# ── Install and start MySQL server on Colab ─────────────────
!apt-get update -qq && apt-get install -y -qq mysql-server > /dev/null 2>&1
!service mysql start
!mysql -e "ALTER USER 'root'@'localhost' IDENTIFIED WITH mysql_native_password BY '';" 2>/dev/null || true
!mysql -e "SELECT 'MySQL is ready!' AS status;"


In [ ]:
# Topic 5 — Enforcement: Orders & Payments Hardening Lab
!pip install -q ipython-sql==0.5.0
!pip install -q pymysql==1.1.0
!pip install -q sqlalchemy==2.0.20
!pip install -q prettytable==2.0.0

%load_ext sql

# Connect to MySQL (no database specified — created in the next cell)
%sql mysql+pymysql://root:@localhost/


**Cell 2 — Create database and base schema:**

In [ ]:
%%sql
DROP DATABASE IF EXISTS orders_payments;
CREATE DATABASE orders_payments;
USE orders_payments;

-- Lookup table (from Chapter 3 BCNF decomposition)
CREATE TABLE Postcode (
    postcode    VARCHAR(10) PRIMARY KEY,
    city        VARCHAR(50) NOT NULL
);

CREATE TABLE Customer (
    customer_id INT PRIMARY KEY,
    name        VARCHAR(100) NOT NULL,
    email       VARCHAR(100) NOT NULL UNIQUE,
    postcode    VARCHAR(10),
    FOREIGN KEY (postcode) REFERENCES Postcode(postcode)
);

CREATE TABLE Order_ (
    order_id       INT PRIMARY KEY,
    customer_id    INT NOT NULL,
    invoice_number VARCHAR(20),
    order_date     DATE,
    status         VARCHAR(20),
    order_total    DECIMAL(10,2),
    FOREIGN KEY (customer_id) REFERENCES Customer(customer_id)
);

CREATE TABLE Product (
    product_id       INT PRIMARY KEY,
    name             VARCHAR(100),
    price            DECIMAL(10,2),
    quantity_in_stock INT DEFAULT 0
);

CREATE TABLE OrderLine (
    order_id   INT NOT NULL,
    product_id INT NOT NULL,
    quantity   INT,
    unit_price DECIMAL(10,2),
    PRIMARY KEY (order_id, product_id),
    FOREIGN KEY (order_id) REFERENCES Order_(order_id),
    FOREIGN KEY (product_id) REFERENCES Product(product_id)
);

CREATE TABLE Payment (
    payment_id   INT PRIMARY KEY,
    order_id     INT NOT NULL UNIQUE,
    amount       DECIMAL(10,2),
    payment_date DATE,
    method       VARCHAR(30),
    FOREIGN KEY (order_id) REFERENCES Order_(order_id)
);

CREATE TABLE Shipping (
    shipping_id       INT PRIMARY KEY,
    order_id          INT NOT NULL UNIQUE,
    shipping_address  VARCHAR(200),
    shipping_city     VARCHAR(50),
    shipping_postcode VARCHAR(10),
    shipped_date      DATE,
    status            VARCHAR(20),
    FOREIGN KEY (order_id) REFERENCES Order_(order_id)
);

CREATE TABLE Review (
    review_id   INT PRIMARY KEY,
    customer_id INT NOT NULL,
    product_id  INT NOT NULL,
    rating      INT,
    comment     TEXT,
    review_date DATE,
    FOREIGN KEY (customer_id) REFERENCES Customer(customer_id),
    FOREIGN KEY (product_id) REFERENCES Product(product_id)
);

CREATE TABLE Supplier (
    supplier_id   INT PRIMARY KEY,
    name          VARCHAR(100),
    contact_email VARCHAR(100)
);

CREATE TABLE SupplierProduct (
    supplier_id INT NOT NULL,
    product_id  INT NOT NULL,
    PRIMARY KEY (supplier_id, product_id),
    FOREIGN KEY (supplier_id) REFERENCES Supplier(supplier_id),
    FOREIGN KEY (product_id) REFERENCES Product(product_id)
);

CREATE TABLE Category (
    category_id INT PRIMARY KEY,
    name        VARCHAR(50),
    description TEXT
);

CREATE TABLE ProductCategory (
    product_id  INT NOT NULL,
    category_id INT NOT NULL,
    PRIMARY KEY (product_id, category_id),
    FOREIGN KEY (product_id) REFERENCES Product(product_id),
    FOREIGN KEY (category_id) REFERENCES Category(category_id)
);


**Cell 3 — Load seed data (reference tables):**

In [ ]:
%%sql
USE orders_payments;

-- Postcodes
INSERT INTO Postcode VALUES
  ('018956','Singapore'),('049315','Singapore'),('50088','Kuala Lumpur'),
  ('50450','Kuala Lumpur'),('10110','Bangkok'),('10500','Bangkok'),
  ('12190','Jakarta'),('10310','Jakarta'),('1000','Manila'),
  ('700000','Ho Chi Minh City');

-- Customers (100)
INSERT INTO Customer (customer_id, name, email, postcode) VALUES
  (1,'Alice Tan','alice@example.com','018956'),
  (2,'Bob Patel','bob@example.com','049315'),
  (3,'Carol Lim','carol@example.com','50088'),
  (4,'Dave Wong','dave@example.com','50450'),
  (5,'Eve Santos','eve@example.com','10110'),
  (6,'Frank Teo','frank@example.com','10500'),
  (7,'Grace Lee','grace@example.com','12190'),
  (8,'Hiro Tanaka','hiro@example.com','10310'),
  (9,'Isla Cruz','isla@example.com','1000'),
  (10,'Jack Nguyen','jack@example.com','700000');

-- Products (50)
INSERT INTO Product (product_id, name, price, quantity_in_stock) VALUES
  (1,'Widget A',9.99,500),(2,'Widget B',14.99,300),(3,'Widget C',19.99,200),
  (4,'Gadget X',49.99,100),(5,'Gadget Y',79.99,80),(6,'Gadget Z',99.99,50),
  (7,'Cable USB-C',4.99,1000),(8,'Cable HDMI',7.99,800),(9,'Adapter DP',12.99,400),
  (10,'Screen 24"',199.99,60),(11,'Screen 27"',299.99,40),(12,'Screen 32"',449.99,25),
  (13,'Keyboard Std',29.99,350),(14,'Keyboard Mech',89.99,150),(15,'Mouse Wireless',24.99,500),
  (16,'Mouse Gaming',59.99,200),(17,'Headset Basic',19.99,400),(18,'Headset Pro',149.99,100),
  (19,'Webcam HD',39.99,250),(20,'Webcam 4K',99.99,80),(21,'Dock USB',79.99,120),
  (22,'Hub USB 4-port',14.99,600),(23,'Charger 65W',34.99,300),(24,'Charger 100W',54.99,150),
  (25,'SSD 500GB',59.99,200),(26,'SSD 1TB',99.99,150),(27,'SSD 2TB',179.99,75),
  (28,'RAM 8GB',29.99,300),(29,'RAM 16GB',54.99,200),(30,'RAM 32GB',99.99,100),
  (31,'Case ATX',49.99,150),(32,'Case mATX',39.99,200),(33,'PSU 550W',59.99,180),
  (34,'PSU 750W',89.99,120),(35,'Fan 120mm',9.99,500),(36,'Fan 140mm',12.99,400),
  (37,'Thermal Paste',7.99,800),(38,'Cable Mgmt Kit',14.99,300),(39,'Mousepad Large',19.99,350),
  (40,'Mousepad XL',29.99,200),(41,'Monitor Arm',39.99,150),(42,'Laptop Stand',24.99,250),
  (43,'Wrist Rest',14.99,400),(44,'Desk Mat',34.99,180),(45,'LED Strip',19.99,500),
  (46,'Surge Protector',24.99,300),(47,'UPS 600VA',99.99,80),(48,'UPS 1000VA',149.99,50),
  (49,'Cleaning Kit',9.99,600),(50,'Tool Kit',19.99,350);

-- Categories
INSERT INTO Category VALUES
  (1,'Peripherals','Input and output devices'),
  (2,'Storage','SSDs, HDDs, and memory'),
  (3,'Cables & Adapters','Connectivity accessories'),
  (4,'Displays','Monitors and screens'),
  (5,'Power','PSUs, chargers, and UPS units');

-- Suppliers
INSERT INTO Supplier VALUES
  (1,'TechParts Pte Ltd','sales@techparts.sg'),
  (2,'ComponentsRUs','orders@componentsrus.com'),
  (3,'CableCo Sdn Bhd','info@cableco.my'),
  (4,'DisplayTech','wholesale@displaytech.com'),
  (5,'PowerSupply Inc','trade@powersupply.com');


**Cell 4 — Generate bulk data (Python):**

In [ ]:
import random
from datetime import date, timedelta

# Connect using sqlalchemy for bulk inserts
from sqlalchemy import create_engine, text
engine = create_engine('mysql+pymysql://root:@localhost/orders_payments')

random.seed(42)

# Generate 90 more customers (IDs 11-100)
customers = []
first_names = ['Wei','Siti','Somchai','Arun','Mei','Putri','Rina','Hafiz',
               'Thanh','Dewi','Jun','Aisha','Kai','Noor','Linh',
               'Farid','Yuki','Ratna','Bao','Priya']
last_names = ['Tan','Lim','Wong','Lee','Ng','Ibrahim','Ahmad','Sato',
              'Nguyen','Patel','Kumar','Chen','Teo','Ong','Ismail',
              'Santoso','Prasert','Huda','Pratama','Suresh']
postcodes = ['018956','049315','50088','50450','10110',
             '10500','12190','10310','1000','700000']

for cid in range(11, 101):
    fn = random.choice(first_names)
    ln = random.choice(last_names)
    customers.append((cid, f'{fn} {ln}', f'{fn.lower()}{cid}@example.com',
                       random.choice(postcodes)))

with engine.connect() as conn:
    conn.execute(text("USE orders_payments"))
    for c in customers:
        conn.execute(text(
            "INSERT INTO Customer (customer_id, name, email, postcode) "
            "VALUES (:cid, :name, :email, :pc)"
        ), {"cid": c[0], "name": c[1], "email": c[2], "pc": c[3]})
    conn.commit()

# Generate 5000 orders with invoice numbers (some duplicates for Q1)
statuses = ['pending','paid','shipped','delivered','completed']
status_weights = [5, 10, 10, 25, 50]  # 90% are shipped/delivered/completed
base_date = date(2023, 1, 1)

orders = []
for oid in range(1, 5001):
    cid = random.randint(1, 100)
    odate = base_date + timedelta(days=random.randint(0, 730))
    st = random.choices(statuses, weights=status_weights, k=1)[0]
    # Invoice numbers: mostly unique, but inject 5 deliberate duplicates
    inv = f'INV-{odate.year}-{oid:05d}'
    orders.append((oid, cid, inv, odate.isoformat(), st))

# Inject 5 duplicate invoice numbers (same customer, same invoice)
# Orders 4996-5000 copy invoice numbers from orders 1-5 (same customer)
for i in range(5):
    src = orders[i]
    oid = 4996 + i
    orders[4995 + i] = (oid, src[1], src[2], orders[4995+i][3], orders[4995+i][4])

with engine.connect() as conn:
    conn.execute(text("USE orders_payments"))
    for batch_start in range(0, len(orders), 500):
        batch = orders[batch_start:batch_start+500]
        for o in batch:
            conn.execute(text(
                "INSERT INTO Order_ (order_id, customer_id, invoice_number, order_date, status) "
                "VALUES (:oid, :cid, :inv, :odate, :st)"
            ), {"oid": o[0], "cid": o[1], "inv": o[2], "odate": o[3], "st": o[4]})
        conn.commit()

# Generate ~50,000 order lines (avg 10 per order)
with engine.connect() as conn:
    conn.execute(text("USE orders_payments"))
    for oid in range(1, 5001):
        n_items = random.randint(1, 20)
        products = random.sample(range(1, 51), min(n_items, 50))
        values = []
        for pid in products:
            qty = random.randint(1, 10)
            price = round(random.uniform(4.99, 199.99), 2)
            conn.execute(text(
                "INSERT INTO OrderLine (order_id, product_id, quantity, unit_price) "
                "VALUES (:oid, :pid, :qty, :price)"
            ), {"oid": oid, "pid": pid, "qty": qty, "price": price})
        if oid % 500 == 0:
            conn.commit()
    conn.commit()

# Generate payments (one per order, ~5000)
with engine.connect() as conn:
    conn.execute(text("USE orders_payments"))
    for oid in range(1, 5001):
        if orders[oid-1][4] != 'pending':  # no payment for pending orders
            amt = round(random.uniform(10, 2000), 2)
            pdate = orders[oid-1][3]
            method = random.choice(['card','paypal','bank_transfer'])
            conn.execute(text(
                "INSERT INTO Payment (payment_id, order_id, amount, payment_date, method) "
                "VALUES (:pid, :oid, :amt, :pdate, :method)"
            ), {"pid": oid, "oid": oid, "amt": amt, "pdate": pdate, "method": method})
        if oid % 500 == 0:
            conn.commit()
    conn.commit()

# Generate shipping records (~4000 — shipped, delivered, completed orders)
with engine.connect() as conn:
    conn.execute(text("USE orders_payments"))
    ship_id = 1
    for oid in range(1, 5001):
        st = orders[oid-1][4]
        if st in ('shipped','delivered','completed'):
            addr = f'{random.randint(1,200)} {random.choice(["Orchard Rd","Jalan Sultan","Sukhumvit Soi","Jalan Sudirman"])}'
            city = random.choice(['Singapore','Kuala Lumpur','Bangkok','Jakarta','Manila'])
            pc = random.choice(postcodes)
            sdate = orders[oid-1][3]
            conn.execute(text(
                "INSERT INTO Shipping (shipping_id, order_id, shipping_address, shipping_city, "
                "shipping_postcode, shipped_date, status) "
                "VALUES (:sid, :oid, :addr, :city, :pc, :sdate, :st)"
            ), {"sid": ship_id, "oid": oid, "addr": addr, "city": city,
                "pc": pc, "sdate": sdate, "st": st})
            ship_id += 1
        if oid % 500 == 0:
            conn.commit()
    conn.commit()

# Generate reviews (~200)
with engine.connect() as conn:
    conn.execute(text("USE orders_payments"))
    for rid in range(1, 201):
        cid = random.randint(1, 100)
        pid = random.randint(1, 50)
        rating = random.randint(1, 5)
        rdate = (base_date + timedelta(days=random.randint(0, 730))).isoformat()
        conn.execute(text(
            "INSERT INTO Review (review_id, customer_id, product_id, rating, comment, review_date) "
            "VALUES (:rid, :cid, :pid, :rating, :comment, :rdate)"
        ), {"rid": rid, "cid": cid, "pid": pid, "rating": rating,
            "comment": f'Review {rid}', "rdate": rdate})
    conn.commit()

# SupplierProduct and ProductCategory
with engine.connect() as conn:
    conn.execute(text("USE orders_payments"))
    for sid in range(1, 6):
        products = random.sample(range(1, 51), 10)
        for pid in products:
            conn.execute(text(
                "INSERT INTO SupplierProduct VALUES (:sid, :pid)"
            ), {"sid": sid, "pid": pid})
    conn.commit()

    # Assign each product to 1-2 categories
    for pid in range(1, 51):
        cats = random.sample(range(1, 6), random.randint(1, 2))
        for cid in cats:
            conn.execute(text(
                "INSERT INTO ProductCategory VALUES (:pid, :cid)"
            ), {"pid": pid, "cid": cid})
    conn.commit()

print("Seed data loaded successfully.")


**Cell 5 — Verify setup:**

In [ ]:
%%sql
USE orders_payments;
SELECT 'Customers' AS entity, COUNT(*) AS cnt FROM Customer
UNION ALL SELECT 'Orders', COUNT(*) FROM Order_
UNION ALL SELECT 'OrderLines', COUNT(*) FROM OrderLine
UNION ALL SELECT 'Payments', COUNT(*) FROM Payment
UNION ALL SELECT 'Shipping', COUNT(*) FROM Shipping
UNION ALL SELECT 'Products', COUNT(*) FROM Product
UNION ALL SELECT 'Reviews', COUNT(*) FROM Review;
-- Expected: ~100 customers, ~5000 orders, ~50000 order lines,
-- ~4750 payments, ~4250 shipping, 50 products, 200 reviews


---

## Part 1 — Add Constraints (Enforcement)

### Q1. Demonstrate missing constraints

The base schema has primary keys and foreign keys — but nothing else. Run the following three inserts. All three should succeed (they are bad data, but the schema does not reject them):

**(a)** Insert an OrderLine with negative quantity:

In [ ]:
%%sql
INSERT INTO OrderLine VALUES (1, 50, -3, 9.99);
-- This is product 50 added to order 1 with quantity -3.
-- Should be rejected (negative quantity is impossible), but currently succeeds.


**(b)** Check: are there duplicate `(customer_id, invoice_number)` pairs in the data?

In [ ]:
%%sql
SELECT customer_id, invoice_number, COUNT(*) AS cnt
FROM Order_
GROUP BY customer_id, invoice_number
HAVING COUNT(*) > 1;


**(c)** Insert a Payment with zero amount:

In [ ]:
%%sql
-- Find an order that has no payment yet
SELECT o.order_id FROM Order_ o
  LEFT JOIN Payment p ON o.order_id = p.order_id
  WHERE p.payment_id IS NULL LIMIT 1;

In [ ]:
%%sql
-- Use the order_id from above (replace <order_id> with the actual value)
-- INSERT INTO Payment (payment_id, order_id, amount, payment_date, method)
-- VALUES (9999, <order_id>, 0.00, '2024-01-01', 'card');
-- amount = 0 makes no business sense, but the schema accepts it.


**(d)** For each insert, explain which business rule is being violated and which constraint type would prevent it.

---

### Q2. Add constraints

**(a)** Add a CHECK constraint on OrderLine to require positive quantity:

In [ ]:
%%sql
ALTER TABLE OrderLine ADD CONSTRAINT chk_quantity CHECK (quantity > 0);


**(b)** First, clean up the bad data from Q1a, then re-run the negative-quantity insert. Record the error message.

**(c)** Add a CHECK constraint on Payment to require positive amount. Clean up Q1c's bad data first, then test.

**(d)** Add a CHECK constraint on Review to ensure `rating BETWEEN 1 AND 5`.

**(e)** Test each constraint: try to insert one invalid row per constraint. All three should fail. Record the error messages.

---

### Q3. Composite UNIQUE constraint

**(a)** *Predict before running:* The data contains deliberate duplicate `(customer_id, invoice_number)` pairs (from the seed generator). What happens if you try to add a UNIQUE constraint while duplicates exist?

**(b)** Remove the duplicate orders (keep the lower order_id, delete the higher one). You will need to delete dependent OrderLine, Payment, and Shipping rows first (FK RESTRICT blocks deletion otherwise).

**(c)** Add the composite UNIQUE constraint:

In [ ]:
%%sql
ALTER TABLE Order_ ADD CONSTRAINT uq_customer_invoice
    UNIQUE (customer_id, invoice_number);


**(d)** Test: attempt to INSERT an order with a `(customer_id, invoice_number)` pair that already exists. Record the error message.

---

### Q4. FK cascade behaviours

**(a)** Try to delete an Order that has OrderLines (with the default RESTRICT behaviour):

In [ ]:
%%sql
DELETE FROM Order_ WHERE order_id = 10;


Record the error message.

**(b)** Change the OrderLine → Order FK to CASCADE:

In [ ]:
%%sql
ALTER TABLE OrderLine DROP FOREIGN KEY orderline_ibfk_1;
ALTER TABLE OrderLine ADD CONSTRAINT fk_orderline_order
    FOREIGN KEY (order_id) REFERENCES Order_(order_id)
    ON DELETE CASCADE;


**(c)** Before deleting, count the OrderLines for order 10:

In [ ]:
%%sql
SELECT COUNT(*) FROM OrderLine WHERE order_id = 10;


**(d)** Now delete order 10:

In [ ]:
%%sql
-- First, remove Payment and Shipping for order 10 (their FKs are still RESTRICT)
DELETE FROM Payment WHERE order_id = 10;
DELETE FROM Shipping WHERE order_id = 10;
-- Now delete order 10 (OrderLine rows cascade automatically)
DELETE FROM Order_ WHERE order_id = 10;


**(e)** Re-count the OrderLines for order 10. Verify they are gone.

**(f)** *Discussion:* Why is CASCADE appropriate for OrderLine → Order (weak entity) but RESTRICT is appropriate for Order → Customer (independent entity)?

---

## Part 2 — Reporting Views (Safe Exposure)

### Q5. Create a `revenue_by_product` view

Write a view that shows: product name, total quantity sold, and total revenue. The view should join Product and OrderLine — no customer data, no payment data.

**(a)** Write and execute the CREATE VIEW statement.

**(b)** Query the view: `SELECT * FROM revenue_by_product ORDER BY total_revenue DESC LIMIT 10;`

**(c)** Verify: run the same query directly against the base tables (without the view). Do the results match?

---

### Q6. Create a `customer_orders` view

Write a view that shows: customer name, order count, and total spend (sum of `order_total` or sum of OrderLine amounts) — but **not** email, not payment details, not internal IDs.

**(a)** Write and execute the CREATE VIEW statement.

**(b)** Query the view. Does it show customer emails? (It should not.)

---

### Q7. Verify views match Chapter 4 queries

**(a)** The `revenue_by_product` view encapsulates the same multi-table join pattern from Chapter 4's FWGHOS pipeline. Write a direct query (without the view) that produces the same result.

**(b)** Compare the two approaches. Which is easier for a reporting user to write? Which exposes fewer implementation details?

---

## Part 3 — Access Control (GRANT and REVOKE)

### Q8. Create users and GRANT permissions

**(a)** Create two users:

In [ ]:
%%sql
CREATE USER IF NOT EXISTS 'app_user'@'localhost' IDENTIFIED BY 'app_pass';
CREATE USER IF NOT EXISTS 'reporting_user'@'localhost' IDENTIFIED BY 'report_pass';


**(b)** Grant `app_user` SELECT, INSERT, UPDATE on all base tables:

In [ ]:
%%sql
GRANT SELECT, INSERT, UPDATE ON orders_payments.* TO 'app_user'@'localhost';


**(c)** Grant `reporting_user` SELECT on the two views only:

In [ ]:
%%sql
-- Note: run this AFTER completing Q5 and Q6 (creating the views)
GRANT SELECT ON orders_payments.revenue_by_product TO 'reporting_user'@'localhost';
GRANT SELECT ON orders_payments.customer_orders TO 'reporting_user'@'localhost';


**(d)** Flush privileges:

In [ ]:
%%sql
FLUSH PRIVILEGES;


---

### Q9. Test reporting_user restrictions

Connect as `reporting_user` (in a new connection or using a new `%sql` connection string).

**(a)** Run `SELECT * FROM revenue_by_product LIMIT 5;` — record the result (should succeed).

**(b)** Run `SELECT * FROM Customer LIMIT 5;` — record the result (should fail: permission denied).

**(c)** Run `UPDATE Product SET price = 0 WHERE product_id = 1;` — record the result (should fail).

**(d)** Summarise: what can `reporting_user` do? What can they not do?

---

### Q10. Test app_user restrictions

Connect as `app_user`.

**(a)** Run `INSERT INTO Order_ (order_id, customer_id, invoice_number, order_date, status) VALUES (9001, 1, 'INV-TEST-001', '2025-01-01', 'pending');` — should succeed.

**(b)** Run `DROP TABLE Customer;` — should fail (no DROP privilege).

**(c)** Clean up the test order.

---

### Q11. Discussion: expanding access safely

The reporting analyst asks for customer emails to send a survey. What is the right response?

**(a)** Should you widen `reporting_user`'s access to include the Customer table? Why or why not?

**(b)** Describe the correct pattern: create a purpose-built view with only the needed columns (name, email — no postcode, no customer_id), create a separate `survey_user`, and GRANT SELECT on that view.

**(c)** Why is creating a new view + user better than modifying existing permissions?

---

## Part 4 — Transactions (Atomic Changes)

### Q12. The no-transaction failure

**(a)** *Predict before running:* You will run two statements without a transaction. The first INSERT succeeds (auto-committed). The second UPDATE fails (targets a non-existent order). What state will the database be in?

**(b)** Run (note: use a payment_id and order_id that do not already have a payment):

In [ ]:
%%sql
-- Find an order without a payment (pending orders)
SELECT order_id FROM Order_ WHERE status = 'pending' LIMIT 1;


Use the order_id from the result:

In [ ]:
%%sql
-- Statement 1: Insert payment (succeeds and auto-commits)
INSERT INTO Payment (payment_id, order_id, amount, payment_date, method)
VALUES (8001, <order_id>, 49.99, '2025-01-15', 'card');

-- Statement 2: Update a NON-EXISTENT order (fails — 0 rows affected)
UPDATE Order_ SET status = 'paid' WHERE order_id = 99999;


**(c)** Check: does the payment exist? Does order 99999 exist? Is this state consistent?

**(d)** This is Failure 3 from the chapter — a payment exists but the order status was never updated. Without a transaction, the first statement is permanent even though the operation as a whole failed.

---

### Q13. Transaction with ROLLBACK

**(a)** Wrap the same logic in a transaction:

In [ ]:
%%sql
BEGIN;

INSERT INTO Payment (payment_id, order_id, amount, payment_date, method)
VALUES (8002, <another_pending_order_id>, 49.99, '2025-01-15', 'card');

-- Simulate failure: update a non-existent order
UPDATE Order_ SET status = 'paid' WHERE order_id = 99999;
-- Check: how many rows were affected? (0 — the order doesn't exist)

-- The developer detects the failure and rolls back
ROLLBACK;


**(b)** *Predict before checking:* After ROLLBACK, does the payment row (8002) exist?

**(c)** Verify: `SELECT * FROM Payment WHERE payment_id = 8002;` — should return zero rows.

**(d)** Explain: why must the developer detect the failure and issue ROLLBACK explicitly in MySQL? (Hint: MySQL does not auto-rollback on a failed UPDATE — the transaction remains open.)

---

### Q14. Successful transaction with COMMIT

**(a)** Write a successful checkout transaction:

In [ ]:
%%sql
BEGIN;

INSERT INTO Payment (payment_id, order_id, amount, payment_date, method)
VALUES (8003, <a_pending_order_id>, 49.99, '2025-01-15', 'card');

UPDATE Order_ SET status = 'paid' WHERE order_id = <same_order_id>;

COMMIT;


**(b)** Verify both changes persist:

In [ ]:
%%sql
SELECT * FROM Payment WHERE payment_id = 8003;
SELECT status FROM Order_ WHERE order_id = <same_order_id>;


**(c)** Compare the three scenarios: no transaction (Q12 — inconsistent), transaction + ROLLBACK (Q13 — clean recovery), transaction + COMMIT (Q14 — consistent completion). Which is appropriate for production code?

---

## Part 5 — Indexes and EXPLAIN (Read Performance)

### Q15. Baseline EXPLAIN

**(a)** Run EXPLAIN on the `revenue_by_product` view query:

In [ ]:
%%sql
EXPLAIN SELECT p.name, SUM(ol.quantity) AS total_qty,
       SUM(ol.quantity * ol.unit_price) AS total_rev
FROM Product p
JOIN OrderLine ol ON p.product_id = ol.product_id
GROUP BY p.product_id, p.name;


**(b)** Identify the `type` column for each table in the EXPLAIN output. Is either table using a full scan (`ALL`)?

**(c)** Record the execution time of the query (use `%%time` or note the time from the `%%sql` output).

---

### Q16. Add indexes and re-EXPLAIN

**(a)** Add an index on `OrderLine.product_id`:

In [ ]:
%%sql
CREATE INDEX idx_orderline_product ON OrderLine(product_id);


**(b)** Add an index on `Order_.order_date`:

In [ ]:
%%sql
CREATE INDEX idx_order_date ON Order_(order_date);


**(c)** Re-run the EXPLAIN from Q15. Has the `type` changed for any table?

**(d)** Re-run the query and compare execution time to Q15c.

---

### Q17. Selectivity prediction

**(a)** *Predict before running:* An index on `Order_.status` — will it help a query that filters `WHERE status = 'completed'`? Consider: approximately what percentage of orders have status `'completed'`?

**(b)** Check the actual distribution:

In [ ]:
%%sql
SELECT status, COUNT(*) AS cnt,
       ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM Order_), 1) AS pct
FROM Order_
GROUP BY status
ORDER BY cnt DESC;


**(c)** Add the index and run EXPLAIN on `SELECT * FROM Order_ WHERE status = 'completed'`:

In [ ]:
%%sql
CREATE INDEX idx_order_status ON Order_(status);
EXPLAIN SELECT * FROM Order_ WHERE status = 'completed';


**(d)** Does the optimiser use the index? If not, why? (Low selectivity — the engine reads most of the table regardless.)

**(e)** Now test with a high-selectivity predicate: `EXPLAIN SELECT * FROM Order_ WHERE status = 'pending';`

Does the optimiser use the index for `'pending'` (which matches ~5% of rows)?

**(f)** Remove the low-value index: `DROP INDEX idx_order_status ON Order_;`

---

### Q18. Chapter 4 loop-back — FWGHOS meets EXPLAIN

Chapter 4 taught you to *read* multi-table queries using the FWGHOS pipeline. This question teaches you to *optimise* them using EXPLAIN.

**(a)** Write a query that finds total revenue by product category for orders placed in 2024:

In [ ]:
%%sql
SELECT c.name AS category, SUM(ol.quantity * ol.unit_price) AS revenue
FROM OrderLine ol
JOIN Product p ON ol.product_id = p.product_id
JOIN ProductCategory pc ON p.product_id = pc.product_id
JOIN Category c ON pc.category_id = c.category_id
JOIN Order_ o ON ol.order_id = o.order_id
WHERE o.order_date BETWEEN '2024-01-01' AND '2024-12-31'
GROUP BY c.category_id, c.name
ORDER BY revenue DESC;


**(b)** Trace the FWGHOS pipeline (from Chapter 4): what does each step produce?

**(c)** Run EXPLAIN on this query. Which tables show full scans?

**(d)** The `idx_order_date` index (from Q16) covers the WHERE clause. Does EXPLAIN show it being used? If the join to Order_ is not the driving table, the optimiser may choose a different plan.

**(e)** *Reflection:* Chapter 4 taught you to understand *what* a query does (FWGHOS trace). This chapter teaches you to understand *how fast* it does it (EXPLAIN). How do the two perspectives complement each other?

---

## Deliverable

- **Part 1** (Q1–Q4): Three constraint violation error messages, CASCADE before/after row counts, discussion of RESTRICT vs CASCADE.
- **Part 2** (Q5–Q7): Two CREATE VIEW statements, view query results, verification against direct queries.
- **Part 3** (Q8–Q11): GRANT statements, permission-denied evidence for `reporting_user` and `app_user`, discussion of safe access expansion.
- **Part 4** (Q12–Q14): No-transaction failure evidence, ROLLBACK verification, successful COMMIT verification, comparison of three scenarios.
- **Part 5** (Q15–Q18): EXPLAIN outputs before and after indexing, selectivity prediction and verification, FWGHOS + EXPLAIN reflection.

---

## Solutions

<details>
<summary><strong>Q1 Solution</strong></summary>

**(a)** The INSERT succeeds — the schema has no CHECK constraint on `quantity`. The OrderLine with `quantity = -3` is stored.

**(b)** The query should show 5 duplicate pairs (injected by the seed generator).

**(c)** The INSERT of a zero-amount payment succeeds (if the order has no existing payment) because there is no CHECK on `amount`.

**(d)**
- Negative quantity → CHECK (quantity > 0)
- Duplicate invoice → UNIQUE (customer_id, invoice_number)
- Zero amount → CHECK (amount > 0)
</details>

---

<details>
<summary><strong>Q2 Solution</strong></summary>

**(b)** Clean up: `DELETE FROM OrderLine WHERE order_id = 1 AND product_id = 50;` (the bad row from Q1a). Then re-insert: `INSERT INTO OrderLine VALUES (1, 50, -3, 9.99);` — now fails with: "Check constraint 'chk_quantity' is violated."

**(e)** Each invalid insert fails:
- `INSERT INTO OrderLine VALUES (2, 1, 0, 5.00)` → chk_quantity violated (0 is not > 0)
- `INSERT INTO Payment VALUES (9998, 2, -10, '2024-01-01', 'card')` → chk_amount violated
- `INSERT INTO Review VALUES (999, 1, 1, 6, 'Great', '2024-01-01')` → chk_rating violated (6 not BETWEEN 1 AND 5)
</details>

---

<details>
<summary><strong>Q3 Solution</strong></summary>

**(a)** The ALTER TABLE fails with a duplicate key error — you cannot add a UNIQUE constraint while duplicate values exist.

**(b)** To clean up duplicates, delete the higher-order_id duplicates and their dependent rows first:

In [ ]:
%%sql
-- For each duplicate pair, find the order_ids to remove
-- Then delete OrderLine, Payment, Shipping rows for those order_ids
-- Then delete the Order_ rows


**(d)** After adding the UNIQUE constraint, attempting to insert a duplicate `(customer_id, invoice_number)` pair fails: "Duplicate entry '...' for key 'uq_customer_invoice'."
</details>

---

<details>
<summary><strong>Q4 Solution</strong></summary>

**(a)** Error: "Cannot delete or update a parent row: a foreign key constraint fails" — RESTRICT blocks the delete because OrderLines reference this Order.

**(c)** The count shows N order lines for order 10 (varies by seed data, typically 1–20).

**(e)** After CASCADE delete, `SELECT COUNT(*) FROM OrderLine WHERE order_id = 10` returns 0 — all line items were automatically deleted.

**(f)** CASCADE is appropriate for OrderLine → Order because OrderLine is a weak entity (its composite PK includes `order_id` — it cannot exist without the order). RESTRICT is appropriate for Order → Customer because Order is an independent entity (it has its own PK). Deleting a customer should not silently destroy their order history.
</details>

---

<details>
<summary><strong>Q5 Solution</strong></summary>

**(a)**

In [ ]:
%%sql
CREATE VIEW revenue_by_product AS
SELECT p.name AS product_name,
       SUM(ol.quantity) AS total_quantity_sold,
       SUM(ol.quantity * ol.unit_price) AS total_revenue
FROM Product p
JOIN OrderLine ol ON p.product_id = ol.product_id
GROUP BY p.product_id, p.name;


**(c)** The view query and the direct query produce identical results — the view is just a named query.
</details>

---

<details>
<summary><strong>Q6 Solution</strong></summary>

**(a)**

In [ ]:
%%sql
CREATE VIEW customer_orders AS
SELECT c.name AS customer_name,
       COUNT(DISTINCT o.order_id) AS order_count,
       COALESCE(SUM(ol.quantity * ol.unit_price), 0) AS total_spend
FROM Customer c
LEFT JOIN Order_ o ON c.customer_id = o.customer_id
LEFT JOIN OrderLine ol ON o.order_id = ol.order_id
GROUP BY c.customer_id, c.name;


Note: LEFT JOIN ensures customers with zero orders still appear. `COUNT(DISTINCT o.order_id)` avoids inflation from OrderLine multiplication (Chapter 4's Failure 1 pattern).

**(b)** The view does not include `email` — it is not in the SELECT list. A user querying this view cannot see customer emails.
</details>

---

<details>
<summary><strong>Q7 Solution</strong></summary>

**(b)** The view is easier — one line: `SELECT * FROM revenue_by_product`. The direct query requires writing the full JOIN and GROUP BY. The view hides implementation details (table names, join conditions) and provides a stable interface even if the underlying schema changes.
</details>

---

<details>
<summary><strong>Q8 Solution</strong></summary>

The GRANT statements give `app_user` broad access to base tables (SELECT, INSERT, UPDATE but not DELETE or DROP) and give `reporting_user` narrow access to views only.
</details>

---

<details>
<summary><strong>Q9 Solution</strong></summary>

**(a)** Succeeds — `reporting_user` has SELECT on the view.

**(b)** Fails: "SELECT command denied to user 'reporting_user'@'localhost' for table 'Customer'."

**(c)** Fails: "UPDATE command denied to user 'reporting_user'@'localhost' for table 'Product'."

**(d)** `reporting_user` can read the two reporting views. They cannot read any base table, cannot insert, update, or delete any data, and cannot modify the schema.
</details>

---

<details>
<summary><strong>Q10 Solution</strong></summary>

**(a)** Succeeds — `app_user` has INSERT on base tables.

**(b)** Fails: "DROP command denied to user 'app_user'@'localhost' for table 'Customer'." The DROP privilege was not granted.
</details>

---

<details>
<summary><strong>Q11 Solution</strong></summary>

**(a)** No — widening `reporting_user`'s access violates least privilege. The analyst does not need access to the full Customer table for a survey. They also do not need payment data, order details, or other sensitive information that `reporting_user` might gain access to in the future.

**(b)** Create a view: `CREATE VIEW survey_contacts AS SELECT name, email FROM Customer;` Create a user: `CREATE USER 'survey_user'@'localhost';` Grant: `GRANT SELECT ON survey_contacts TO 'survey_user'@'localhost';`

**(c)** Creating a new view + user isolates the access. If the survey project ends, you revoke (or drop) the `survey_user` — no other permissions are affected. Modifying existing permissions risks over-granting and creates maintenance complexity.
</details>

---

<details>
<summary><strong>Q12 Solution</strong></summary>

**(a)** The payment exists (auto-committed). The order status is unchanged (the UPDATE affected 0 rows — order 99999 does not exist). This state is inconsistent: a payment exists for a status that was never updated.

**(d)** This is Failure 3: individually valid statements produce inconsistent state when one succeeds and the other fails. Without a transaction, there is no way to undo the first statement.
</details>

---

<details>
<summary><strong>Q13 Solution</strong></summary>

**(b)** No — the payment row does not exist after ROLLBACK. The transaction undid all changes since BEGIN.

**(c)** Zero rows returned — confirmed.

**(d)** MySQL does not auto-rollback on a non-fatal error (like a zero-row UPDATE). The transaction remains open. The developer must check the affected row count and issue ROLLBACK if the operation did not complete as expected. Some frameworks handle this automatically, but the underlying behaviour is: the developer is responsible for detecting partial failure within a transaction.
</details>

---

<details>
<summary><strong>Q14 Solution</strong></summary>

**(b)** Both rows exist: the payment is stored and the order status is 'paid'. COMMIT made both changes permanent.

**(c)** Production code should always use transactions for multi-step operations (Q14 pattern). Q12 (no transaction) is dangerous — partial failure leaves inconsistent state. Q13 (transaction + ROLLBACK) is the correct recovery path when something goes wrong. Q14 (transaction + COMMIT) is the normal success path.
</details>

---

<details>
<summary><strong>Q15 Solution</strong></summary>

**(b)** Without indexes (beyond PK), the EXPLAIN likely shows `type: ALL` for OrderLine (full scan on ~50,000 rows) and `type: ALL` or `type: index` for Product.

**(c)** Execution time depends on hardware, but should be measurable (tens to hundreds of milliseconds on 50K rows).
</details>

---

<details>
<summary><strong>Q16 Solution</strong></summary>

**(c)** After adding `idx_orderline_product`, the EXPLAIN for the join `ON ol.product_id = p.product_id` should show `type: ref` for OrderLine (index lookup instead of full scan).

**(d)** Execution time should decrease — the index avoids scanning all 50,000 OrderLine rows for each product.
</details>

---

<details>
<summary><strong>Q17 Solution</strong></summary>

**(a)** An index on `status` is unlikely to help for `WHERE status = 'completed'` because ~50% of orders are 'completed' (low selectivity). The engine still reads most of the table.

**(d)** The optimiser likely ignores the index for `'completed'` (full scan is cheaper than 50,000 index lookups). The EXPLAIN may still show `type: ALL`.

**(e)** For `'pending'` (~5% of rows), the optimiser is more likely to use the index (`type: ref` or `type: range`) because only a small fraction of rows match — the index provides a genuine benefit.
</details>

---

<details>
<summary><strong>Q18 Solution</strong></summary>

**(b)** FWGHOS trace:
- **F:** OrderLine (~50K) ⋈ Product (many-to-one, preserved) ⋈ ProductCategory (one-to-many, may multiply) ⋈ Category (many-to-one, preserved) ⋈ Order_ (many-to-one, preserved). Result: ~50K+ rows.
- **W:** `WHERE order_date BETWEEN '2024-01-01' AND '2024-12-31'` filters to ~25K rows (one year of two).
- **G:** GROUP BY category → ~5 groups.
- **H:** No HAVING.
- **O:** ORDER BY revenue DESC.
- **S:** Output category name and SUM.

**(d)** Whether `idx_order_date` is used depends on the optimiser's plan. If Order_ is not the driving table, the index may not appear in the plan. The optimiser chooses the most efficient overall strategy, not necessarily the one that uses every available index.

**(e)** FWGHOS tells you *what* the query produces (how many rows survive each stage, what the groups look like). EXPLAIN tells you *how efficiently* the engine retrieves those rows (full scan vs index, join strategy, estimated cost). Together they give a complete picture: correctness (FWGHOS) and performance (EXPLAIN).
</details>